# Predicting Washington Public Bike Rentals

The dataset for this guided project is from Capital Bike Share, a bike sharing service in Washington. The dataset recorded the number of bikes rented each hour, if the renter was a casual user or registered, as well as weather data such as temperature and humidity.

The goal of this assignment was to practice using Linear Regression, Decision Trees, and Random Forests using the SKLearn machine learning library. 

We begin by loading in the data and doing some quick exploration.

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt


bike_rentals = pd.read_csv('bike_rental_hour.csv')
print(bike_rentals.head())

plt.hist(bike_rentals['cnt'])
plt.show()

print(bike_rentals.corr()['cnt'])


We will do some feature engineering before creating the machine learning models. We will collapse the 24 hour time period into 4 six-hour groups. 

In [ ]:
# Creating (numeric) labels to divide the day into 4 parts
def assign_label(hour):
    if 0 <= hour < 6:
        return 4
    elif 6 <= hour < 12:
        return 1
    elif 12 <= hour < 18:
        return 2
    elif 18 <= hour <= 24:
        return 3
    
bike_rentals['time_label'] = bike_rentals['hr'].apply(assign_label)

# Linear Regression
Before training the machine learning model, we will split the data into train and test sets. A list of features will be created from the columns of the bike rental dataset with our label ("cnt") and other useless features dropped. 

With the train and test sets ready and the features selected, the linear regression model can will then be created, trained, and tested. The error metric used for all the machine learning models in this assignment will be mean squared error.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Creating a test and training set
train = bike_rentals.sample(frac=.8)
test = bike_rentals[~bike_rentals.index.isin(train.index)]

# Preparing the columns wanted for the regression
predictors = list(bike_rentals.columns)
predictors.remove("cnt")
predictors.remove("casual")
predictors.remove("registered")
predictors.remove("dteday")

lr = LinearRegression()
lr.fit(train[predictors], train['cnt'])
predictions = lr.predict(test[predictors])
mse = mean_squared_error(test['cnt'], predictions)
print('\n')
print("Mean Squared Error: ",mse)
print("Root Mean Squared Error: ", mse**.5)

# Decision Trees
The train and test sets will now be applied to a decision tree model.  After the initial model is trained and tested, another decision tree will be modeled with some tinkering of the hyper-parameters with the hopes of tuning the model for more accuracy.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import roc_auc_score

# Instantiate a decision tree regressor. Train. Test. 
dt = DecisionTreeRegressor()
dt.fit(train[predictors], train['cnt'])
dt_predictions = dt.predict(test[predictors])

mse_dt = mean_squared_error(test['cnt'], dt_predictions)
print('\n')
print("Results of Decision Tree")
print("Mean Squared Error: ",mse_dt)
print("Root Mean Squared Error: ", mse_dt**.5)

# Tinker with the regressor to improve accuracy
dt = DecisionTreeRegressor(min_samples_leaf=15, max_depth=25)
dt.fit(train[predictors], train['cnt'])
dt_predictions = dt.predict(test[predictors])

mse_dt = mean_squared_error(test['cnt'], dt_predictions)
print('\n')
print("Result of Tweaked Decision Tree")
print("Mean Squared Error: ",mse_dt)
print("Root Mean Squared Error: ", mse_dt**.5)

# Random Forests
Similiarly to the decision trees above, this random forest model will be created twice with the second iteration having it's hyper-parameters tuned. An added avenue of exploration here is using the trained model to test itself against the training data it saw while it was being fit. The difference or similarity between the error scores could be an indication that our model is overfitting. 

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Create a random forest and fit it
rf = RandomForestRegressor()
rf.fit(train[predictors], train['cnt'])
# Create predictions on test data
rf_predictions = rf.predict(test[predictors])
mse_rf = mean_squared_error(test['cnt'], rf_predictions)
# Create predictions on training data to test for overfitting
rf_predictions_train = rf.predict(train[predictors])
mse_rf_training = mean_squared_error(train['cnt'], rf_predictions_train)
print('\n')
print("Result of Random Forest")
print("Predictions Using Test Data")
print("Mean Squared Error: ",mse_rf)
print("Root Mean Squared Error: ", mse_rf**.5)
print("Predictions Using Training Data")
print("Mean Squared Error: ",mse_rf_training)
print("Root Mean Squared Error: ", mse_rf_training**.5)

# Tweak the Random Forest 
rf = RandomForestRegressor(min_samples_leaf=27, max_depth=17)
rf.fit(train[predictors], train['cnt'])
rf_predictions = rf.predict(test[predictors])
mse_rf = mean_squared_error(test['cnt'], rf_predictions)
rf_predictions_train = rf.predict(train[predictors])
mse_rf_training = mean_squared_error(train['cnt'], rf_predictions_train)
        
print('\n')
print("Result of Tweaked Random Forest")
print("Predictions Using Test Data")
print("Mean Squared Error: ",mse_rf)
print("Root Mean Squared Error: ", mse_rf**.5)
print("Predictions Using Training Data")
print("Mean Squared Error: ",mse_rf_training)
print("Root Mean Squared Error: ", mse_rf_training**.5)

In conclusion we can see that the Random Forest algorithm outperformed both Linear Regression and the Decision Tree. Although the tweaked Random Forest performed worse, the result is not alarming. A better tuning could be found if extra time is spent tuning the hyper-parameters. 